In [0]:
"""
04_material_kpis.py

Manufacturing Material KPIs

Source:
    fact_materials

Target:
    material_kpis

Author:
Sumanth Vempalle

Version:
2.0.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    sum,
    when,
)

# ============================================================
# Material KPIs
# ============================================================

@dlt.table(
    name="material_kpis",
    comment="Manufacturing material consumption and scan KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def material_kpis():

    materials = dlt.read("fact_materials")

    return (

        materials

        .groupBy(

            "plant_code",
            "supplier",
            "material_number",
            "product_code",
            "product_name",

        )

        .agg(

            count("*").alias(
                "materials_scanned"
            ),

            sum(

                when(

                    col("scan_status") == "SUCCESS",

                    1

                ).otherwise(0)

            ).alias(
                "successful_scans"
            ),

            (
                count("*")

                -

                sum(

                    when(

                        col("scan_status") == "SUCCESS",

                        1

                    ).otherwise(0)

                )

            ).alias(
                "failed_scans"
            ),

        )

        .withColumn(

            "scan_success_rate",

            when(

                col("materials_scanned") > 0,

                col("successful_scans")
                * 100.0
                / col("materials_scanned")

            ).otherwise(0.0)

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )